# Model 3: Custom FNet, Perceiver & ResNeXt50 Architectures

This notebook demonstrates custom Fourier Mixing (**FNet**), Cross-Attention (**Perceiver**), and **ResNeXt50** architectures built from scratch for lightweight multi-class eye disease prediction.

In [ ]:
import os
import torch
import torch.nn as nn

CLASSES = ['AMD', 'Cataract', 'Dementia', 'Diabetes', 'Glaucoma', 'Normal']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHT_FNET = 'weights/fnet_scratch_best.pth'
WEIGHT_PERCEIVER = 'weights/perceiver_scratch_best.pth'
WEIGHT_RESNEXT = 'weights/resnext50_scratch_best.pth'

print(f"Device set to: {DEVICE}")

## 1. Custom FNet Implementation (Fourier Transform Mixing)

In [ ]:
class FNetBlock(nn.Module):
    def __init__(self, dim=256, ff_dim=1024):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, dim)
        )

    def forward(self, x):
        x = x + torch.fft.fft(torch.fft.fft(self.ln1(x), dim=-1), dim=-2).real
        x = x + self.ff(self.ln2(x))
        return x

class FNet(nn.Module):
    def __init__(self, img_size=640, patch_size=32, embed_dim=256, num_blocks=6, num_classes=6):
        super().__init__()
        self.patch_size = patch_size
        num_patches = (img_size // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size
        self.embedding = nn.Linear(patch_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.blocks = nn.Sequential(*[FNetBlock(embed_dim) for _ in range(num_blocks)])
        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.view(B, C, H // p, p, W // p, p).permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C * p * p)
        x = self.embedding(x) + self.pos_embed
        x = self.blocks(x)
        x = self.ln(x)
        x = x.mean(dim=1)
        return self.head(x)

fnet_model = FNet()
if os.path.exists(WEIGHT_FNET):
    fnet_model.load_state_dict(torch.load(WEIGHT_FNET, map_location=DEVICE))
    print("FNet weights loaded successfully.")

## 2. Multi-Model Inference Benchmark

In [ ]:
fnet_model.eval().to(DEVICE)
test_tensor = torch.randn(1, 3, 640, 640).to(DEVICE)
with torch.no_grad():
    out = fnet_model(test_tensor)
    print("FNet Output Logits:", out)